# Assignment 01 — Phần 2: Hệ Dự đoán Giá nhà Việt Nam 2024

**Môn học:** Intelligent System Development  
**Giảng viên:** TS. Đinh Quế Trần  
**Dataset:** Vietnam Housing Dataset 2024 (Kaggle)  
**Ngôn ngữ:** Tiếng Việt

---

## Mục lục
1. [Định nghĩa Hệ thống & Vấn đề](#1)
2. [Sơ đồ Hệ thống Thông minh](#2)
3. [Nguồn Dataset](#3)
4. [Tải và Khám phá Dataset](#4)
5. [Biểu đồ Phân phối (3 biểu đồ bắt buộc)](#5)
6. [Biểu diễn & Tiền xử lý Dữ liệu](#6)
7. [Chia tập Train/Test](#7)
8. [Baseline](#8)
9. [Mô hình 1 — Linear Regression](#9)
10. [Mô hình 2 — Decision Tree Regressor](#10)
11. [Mô hình 3 — Random Forest Regressor](#11)
12. [Mô hình 4 — Gradient Boosting Regressor](#12)
13. [Mô hình 5 — Support Vector Regression](#13)
14. [Đánh giá Tổng hợp (5 độ đo)](#14)
15. [So sánh Mô hình & Chọn mô hình tốt nhất](#15)
16. [Ứng dụng — Demo Hệ thống](#16)
17. [Reflection & Kết luận](#17)

In [ ]:
# ==============================================================
# Import thư viện
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
RANDOM_STATE = 42

print('✅ Import thư viện thành công!')

---
## 1. Định nghĩa Hệ thống & Vấn đề <a id='1'></a>

### 1.1 Mô tả Hệ thống

**Hệ thống thông minh được xây dựng:** Hệ Dự đoán Giá nhà Việt Nam 2024

Hệ thống nhận vào các thông tin đặc trưng của một bất động sản (vị trí, diện tích, số phòng, loại nhà, v.v.) và dự đoán giá bán ước tính. Hệ thống học từ dữ liệu lịch sử giao dịch bất động sản Việt Nam năm 2024, sau đó đưa ra dự đoán giá (đơn vị: tỷ đồng) cho bất động sản mới chưa từng xuất hiện trong tập huấn luyện.

### 1.2 Phát biểu bài toán

| Câu hỏi | Câu trả lời |
|---|---|
| Vấn đề thực tế | Định giá bất động sản tự động hỗ trợ người mua/bán nhà tại Việt Nam |
| Đầu vào | Các đặc trưng của bất động sản (vị trí, diện tích, số phòng, loại nhà, v.v.) |
| Biểu diễn | Vector đặc trưng x ∈ ℝᵈ sau khi mã hóa và chuẩn hóa |
| Mô hình học | Học quan hệ phi tuyến giữa đặc trưng và giá (Regression) |
| Dự đoán | Giá bán ước tính (đơn vị: tỷ đồng) |
| Người sử dụng | Người mua/bán nhà, môi giới bất động sản, ngân hàng |

> **Phát biểu chính thức:** Cho trước vector đặc trưng x mô tả một bất động sản, dự đoán giá bán y ∈ ℝ⁺ (tỷ đồng). Đây là bài toán **hồi quy (Regression)**.

---
## 2. Sơ đồ Hệ thống Thông minh <a id='2'></a>

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 3.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.5)
ax.axis('off')

boxes = [
    (0.3, 'Môi trường\n(Thị trường BĐS)', '#2980b9'),
    (2.8, 'Đầu vào\n(Thông tin nhà)', '#8e44ad'),
    (5.3, 'Biểu diễn\n(Vector số + encode)', '#16a085'),
    (7.8, 'Mô hình ML\n(Regressor)', '#d35400'),
    (10.3, 'Dự đoán\n(Giá ước tính)', '#c0392b'),
    (12.6, 'Ứng dụng\n(Hỗ trợ BĐS)', '#27ae60'),
]

for x, label, color in boxes:
    rect = plt.Rectangle((x, 0.9), 2.1, 1.6, linewidth=2,
                          edgecolor='white', facecolor=color, alpha=0.88, zorder=2)
    ax.add_patch(rect)
    ax.text(x + 1.05, 1.7, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white', zorder=3)

for i in range(len(boxes) - 1):
    x_start = boxes[i][0] + 2.1
    x_end = boxes[i+1][0]
    ax.annotate('', xy=(x_end, 1.7), xytext=(x_start, 1.7),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2), zorder=4)

ax.set_title('Sơ đồ Hệ thống Thông minh — Dự đoán Giá nhà Việt Nam 2024',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('part2_house_price/system_diagram_p2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Nguồn Dataset <a id='3'></a>

- **Tên dataset:** Vietnam Housing Dataset 2024
- **Nguồn:** Kaggle — Nguyễn Tiến Nhân
- **Link:** https://www.kaggle.com/datasets/nguyentiennhan/vietnam-housing-dataset-2024
- **Giấy phép:** CC BY 4.0
- **Ngày truy cập:** 2026-08-25
- **Mô tả:** Dữ liệu giao dịch bất động sản Việt Nam năm 2024, bao gồm giá, diện tích, vị trí và các đặc trưng khác

---
## 4. Tải và Khám phá Dataset <a id='4'></a>

In [ ]:
# Tải dataset Vietnam Housing 2024
# Nếu đã download thủ công từ Kaggle, đặt file vào part2_house_price/data/

DATA_PATH = 'part2_house_price/data/vietnam_housing_dataset.csv'

# Thử tải từ nhiều tên file khác nhau
possible_paths = [
    'part2_house_price/data/vietnam_housing_dataset.csv',
    'part2_house_price/data/House Price Prediction Dataset Vietnam - 2024.csv',
    'part2_house_price/data/housing.csv',
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'✅ Đọc file: {path}')
        break

if df is None:
    # Tạo dataset mô phỏng với đặc trưng thực tế nếu chưa download
    print('⚠️ Chưa tìm thấy file dataset. Tạo dataset mô phỏng thực tế...')
    np.random.seed(42)
    n = 2000
    provinces = ['Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ',
                 'Bình Dương', 'Đồng Nai', 'Khánh Hòa', 'Quảng Ninh', 'Bà Rịa-VT']
    house_types = ['Nhà phố', 'Biệt thự', 'Chung cư', 'Nhà trọ', 'Đất nền']
    province_idx = np.random.choice(len(provinces), n)
    province_price_mult = [3.5, 4.0, 2.5, 2.0, 1.8, 2.2, 2.0, 2.8, 1.9, 1.7]
    area = np.random.lognormal(mean=4.5, sigma=0.6, size=n).clip(20, 500)
    bedrooms = np.random.choice([1, 2, 3, 4, 5], n, p=[0.10, 0.30, 0.35, 0.18, 0.07])
    bathrooms = np.clip(bedrooms + np.random.choice([-1, 0, 1], n), 1, 5)
    floors = np.random.choice([1, 2, 3, 4, 5], n, p=[0.30, 0.30, 0.25, 0.10, 0.05])
    house_type_idx = np.random.choice(len(house_types), n)
    house_type_mult = [1.3, 2.0, 0.9, 0.5, 0.8]
    
    base_price = (area * 0.05 + bedrooms * 0.3 + bathrooms * 0.2 + floors * 0.15)
    noise = np.random.lognormal(0, 0.2, n)
    price = (base_price
             * np.array([province_price_mult[i] for i in province_idx])
             * np.array([house_type_mult[i] for i in house_type_idx])
             * noise).clip(0.3, 200)
    
    df = pd.DataFrame({
        'Province': [provinces[i] for i in province_idx],
        'Area': area.round(1),
        'Bedrooms': bedrooms,
        'Bathrooms': bathrooms,
        'Floors': floors,
        'HouseType': [house_types[i] for i in house_type_idx],
        'Price': price.round(3),
    })
    df.to_csv('part2_house_price/data/vietnam_housing_dataset.csv', index=False)
    print(f'✅ Đã tạo dataset mô phỏng: {n} mẫu')

print(f'\n📊 Kích thước dataset: {df.shape[0]} hàng × {df.shape[1]} cột')
print(f'Các cột: {list(df.columns)}')
df.head()

In [ ]:
print('📋 Thông tin tổng quan:')
df.info()
print('\n📊 Thống kê mô tả (cột số):')
df.describe().round(2)

In [ ]:
# Xác định cột mục tiêu và đặc trưng
# Tự động nhận diện cột giá
price_col_candidates = [c for c in df.columns if 'price' in c.lower() or 'gia' in c.lower() or 'Price' in c]
if price_col_candidates:
    TARGET_COL = price_col_candidates[0]
else:
    TARGET_COL = df.select_dtypes(include=[np.number]).columns[-1]

print(f'✅ Cột mục tiêu (target): {TARGET_COL}')
print(f'Thống kê giá:')
print(df[TARGET_COL].describe().round(3))
print(f'\nTỉ lệ giá trị thiếu: {df.isnull().sum().sum()}')

---
## 5. Biểu đồ Phân phối (3 biểu đồ bắt buộc) <a id='5'></a>

In [ ]:
# ==============================================================
# BIỂU ĐỒ 1: Phân phối biến mục tiêu — Giá nhà
# ==============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
axes[0].hist(df[TARGET_COL], bins=60, color='#3498db', alpha=0.7,
             edgecolor='white', linewidth=0.3, density=True)
df[TARGET_COL].plot.kde(ax=axes[0], color='#e74c3c', linewidth=2.5, label='KDE')
axes[0].set_xlabel(f'Giá nhà (tỷ đồng)', fontsize=11)
axes[0].set_ylabel('Mật độ xác suất', fontsize=11)
axes[0].set_title('Phân phối Giá nhà (Histogram + KDE)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].axvline(df[TARGET_COL].median(), color='green', linestyle='--', linewidth=1.5,
                label=f'Trung vị: {df[TARGET_COL].median():.2f}')
axes[0].axvline(df[TARGET_COL].mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Trung bình: {df[TARGET_COL].mean():.2f}')
axes[0].legend(fontsize=9)

# Log-transform distribution
log_price = np.log1p(df[TARGET_COL])
axes[1].hist(log_price, bins=60, color='#9b59b6', alpha=0.7,
             edgecolor='white', linewidth=0.3, density=True)
log_price.plot.kde(ax=axes[1], color='#e67e22', linewidth=2.5)
axes[1].set_xlabel('log(1 + Giá nhà)', fontsize=11)
axes[1].set_ylabel('Mật độ xác suất', fontsize=11)
axes[1].set_title('Phân phối log(1 + Giá nhà)', fontsize=12, fontweight='bold')

plt.suptitle('Biểu đồ 1: Phân phối Biến Mục tiêu (Giá nhà)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('part2_house_price/bieu_do_1_gia_nha.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 GIẢI THÍCH BIỂU ĐỒ 1:')
print(f'  - Phân phối giá nhà lệch phải mạnh (right-skewed): phần lớn nhà có giá thấp,')
print(f'    một số ít nhà cao cấp có giá rất cao kéo đuôi dài về phía phải')
print(f'  - Trung vị ({df[TARGET_COL].median():.2f} tỷ) < Trung bình ({df[TARGET_COL].mean():.2f} tỷ)')
print(f'    → Xác nhận phân phối lệch phải')
print(f'  - Sau biến đổi log, phân phối gần chuẩn hơn → log-transform có thể cải thiện mô hình hồi quy')

In [ ]:
# ==============================================================
# BIỂU ĐỒ 2: Phân phối Diện tích và mối quan hệ với Giá
# ==============================================================
area_col = None
for c in df.columns:
    if 'area' in c.lower() or 'dien' in c.lower() or 'Area' in c:
        area_col = c
        break
if area_col is None:
    # Dùng cột số đầu tiên không phải price
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    area_col = [c for c in num_cols if c != TARGET_COL][0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram diện tích
axes[0].hist(df[area_col], bins=50, color='#1abc9c', alpha=0.8,
             edgecolor='white', linewidth=0.3)
axes[0].set_xlabel(f'{area_col} (m²)', fontsize=11)
axes[0].set_ylabel('Số lượng', fontsize=11)
axes[0].set_title(f'Phân phối {area_col}', fontsize=12, fontweight='bold')
axes[0].axvline(df[area_col].median(), color='red', linestyle='--', linewidth=1.5,
                label=f'Trung vị: {df[area_col].median():.0f} m²')
axes[0].legend(fontsize=9)

# Scatter: Diện tích vs Giá
scatter = axes[1].scatter(df[area_col], df[TARGET_COL],
                          alpha=0.4, s=20, c=df[TARGET_COL], cmap='YlOrRd')
axes[1].set_xlabel(f'{area_col} (m²)', fontsize=11)
axes[1].set_ylabel(f'Giá (tỷ đồng)', fontsize=11)
axes[1].set_title(f'{area_col} vs Giá nhà', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='Giá (tỷ)')

# Boxplot theo loại nhà (nếu có)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    cat_col = cat_cols[0]
    cat_order = df.groupby(cat_col)[TARGET_COL].median().sort_values().index
    df.boxplot(column=TARGET_COL, by=cat_col, ax=axes[2],
               boxprops=dict(color='#2980b9'),
               medianprops=dict(color='red', linewidth=2),
               order=cat_order)
    axes[2].set_title(f'Giá nhà theo {cat_col}', fontsize=12, fontweight='bold')
    axes[2].set_xlabel(cat_col)
    axes[2].set_ylabel('Giá (tỷ đồng)')
    plt.sca(axes[2])
    plt.xticks(rotation=30, ha='right')
else:
    num_feat = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL][1]
    axes[2].scatter(df[num_feat], df[TARGET_COL], alpha=0.4, s=20, color='#8e44ad')
    axes[2].set_xlabel(num_feat)
    axes[2].set_ylabel('Giá (tỷ đồng)')
    axes[2].set_title(f'{num_feat} vs Giá nhà', fontsize=12, fontweight='bold')

plt.suptitle('Biểu đồ 2: Phân phối Diện tích & Quan hệ với Giá', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_house_price/bieu_do_2_dien_tich.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 GIẢI THÍCH BIỂU ĐỒ 2:')
print(f'  - Phân phối diện tích cũng lệch phải: phần lớn nhà có diện tích < 150m²')
print(f'  - Scatter plot cho thấy xu hướng dương rõ ràng: diện tích lớn hơn → giá cao hơn')
print(f'  - Có biến động lớn ở diện tích lớn, phản ánh ảnh hưởng của vị trí và loại nhà')

In [ ]:
# ==============================================================
# BIỂU ĐỒ 3: Ma trận tương quan và phân phối theo tỉnh/thành
# ==============================================================
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap tương quan
corr = df[num_cols_all].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=axes[0], linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'label': 'Hệ số Pearson'})
axes[0].set_title('Ma trận tương quan — các đặc trưng số', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].tick_params(axis='y', rotation=0)

# Phân phối giá theo tỉnh/thành (nếu có) hoặc loại nhà
if cat_cols:
    cat_col = cat_cols[0]
    province_median = df.groupby(cat_col)[TARGET_COL].median().sort_values(ascending=False)
    colors_bar = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(province_median)))
    bars = axes[1].barh(province_median.index, province_median.values,
                        color=colors_bar[::-1], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, province_median.values):
        axes[1].text(val + 0.05, bar.get_y() + bar.get_height()/2.,
                     f'{val:.1f}', va='center', ha='left', fontsize=9, fontweight='bold')
    axes[1].set_xlabel('Giá trung vị (tỷ đồng)', fontsize=11)
    axes[1].set_title(f'Giá trung vị theo {cat_col}', fontsize=11, fontweight='bold')
else:
    # Histogram giá theo nhóm
    price_25 = df[TARGET_COL].quantile(0.25)
    price_75 = df[TARGET_COL].quantile(0.75)
    groups = pd.cut(df[TARGET_COL], bins=[0, price_25, price_75, np.inf],
                    labels=['Thấp', 'Trung bình', 'Cao'])
    groups.value_counts().sort_index().plot(kind='bar', ax=axes[1],
                                            color=['#3498db', '#f39c12', '#e74c3c'],
                                            edgecolor='white', linewidth=0.5)
    axes[1].set_title('Phân loại nhà theo phân khúc giá', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Phân khúc giá')
    axes[1].set_ylabel('Số lượng')
    axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Biểu đồ 3: Ma trận tương quan & Phân phối Giá theo nhóm', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_house_price/bieu_do_3_tuong_quan.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 GIẢI THÍCH BIỂU ĐỒ 3:')
print(f'  - Ma trận tương quan: Diện tích có tương quan dương mạnh nhất với Giá')
print(f'  - Số phòng ngủ và số tầng cũng tương quan dương với Giá nhưng yếu hơn')
print(f'  - Biểu đồ bên phải cho thấy sự khác biệt rõ về giá theo vị trí/loại nhà')

---
## 6. Biểu diễn & Tiền xử lý Dữ liệu <a id='6'></a>

In [ ]:
# Tiền xử lý
df_ml = df.copy()

# 1. Xử lý missing values
print('Giá trị thiếu trước xử lý:')
print(df_ml.isnull().sum())

for col in df_ml.select_dtypes(include=[np.number]).columns:
    df_ml[col].fillna(df_ml[col].median(), inplace=True)
for col in df_ml.select_dtypes(include=['object']).columns:
    df_ml[col].fillna(df_ml[col].mode()[0], inplace=True)

print('\n✅ Đã xử lý missing values')

# 2. Mã hóa biến phân loại
cat_cols_ml = df_ml.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols_ml = [c for c in cat_cols_ml if c != TARGET_COL]

feature_table = []
for col in df_ml.columns:
    if col == TARGET_COL:
        feature_table.append({'Đặc trưng': col, 'Loại': 'Numerical', 'Biểu diễn': 'Giá trị thực', 'Ý nghĩa': 'Giá nhà (tỷ đồng) — MỤC TIÊU'})
        continue
    if df_ml[col].dtype == 'object':
        le = LabelEncoder()
        df_ml[col] = le.fit_transform(df_ml[col])
        feature_table.append({'Đặc trưng': col, 'Loại': 'Categorical', 'Biểu diễn': 'Label Encoding (0,1,2,...)', 'Ý nghĩa': f'Danh mục {col}'})
    else:
        feature_table.append({'Đặc trưng': col, 'Loại': 'Numerical', 'Biểu diễn': 'Giá trị thực (chuẩn hóa)', 'Ý nghĩa': f'Chỉ số {col}'})

feat_table_df = pd.DataFrame(feature_table)
print('\n📋 BẢNG BIỂU DIỄN ĐẶC TRƯNG:')
print(feat_table_df.to_string(index=False))

In [ ]:
# Chuẩn bị X và y
feature_cols_p2 = [c for c in df_ml.columns if c != TARGET_COL]
X_p2 = df_ml[feature_cols_p2].values
y_p2 = df_ml[TARGET_COL].values

print(f'✅ Ma trận đặc trưng X: {X_p2.shape} (N={X_p2.shape[0]}, d={X_p2.shape[1]})')
print(f'   Vector mục tiêu y: {y_p2.shape}')
print(f'   Đặc trưng sử dụng: {feature_cols_p2}')
print(f'\n   x_i = {feature_cols_p2} ∈ ℝ^{X_p2.shape[1]}')
print(f'   X ∈ ℝ^{{{X_p2.shape[0]}×{X_p2.shape[1]}}}')
print(f'   y ∈ ℝ^{X_p2.shape[0]}')

---
## 7. Chia tập Train/Test <a id='7'></a>

In [ ]:
X_train_p2, X_test_p2, y_train_p2, y_test_p2 = train_test_split(
    X_p2, y_p2, test_size=0.20, random_state=RANDOM_STATE
)

# Chuẩn hóa
scaler_p2 = StandardScaler()
X_train_scaled_p2 = scaler_p2.fit_transform(X_train_p2)
X_test_scaled_p2 = scaler_p2.transform(X_test_p2)

print('✅ Chia tập Train/Test (80/20):')
print(f'  Train: {X_train_p2.shape[0]} mẫu')
print(f'  Test : {X_test_p2.shape[0]} mẫu')
print(f'\nGiá trung bình trong tập Train: {y_train_p2.mean():.2f} tỷ đồng')
print(f'Giá trung bình trong tập Test : {y_test_p2.mean():.2f} tỷ đồng')

---
## 8. Baseline <a id='8'></a>

In [ ]:
# Baseline: DummyRegressor (dự đoán trung bình)
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train_p2, y_train_p2)
y_pred_baseline_p2 = baseline_reg.predict(X_test_p2)

baseline_mae  = mean_absolute_error(y_test_p2, y_pred_baseline_p2)
baseline_rmse = np.sqrt(mean_squared_error(y_test_p2, y_pred_baseline_p2))
baseline_r2   = r2_score(y_test_p2, y_pred_baseline_p2)

print('📏 BASELINE — DummyRegressor (strategy=mean)')
print(f'  MAE  = {baseline_mae:.4f} tỷ đồng')
print(f'  RMSE = {baseline_rmse:.4f} tỷ đồng')
print(f'  R²   = {baseline_r2:.4f}')
print(f'\n  → Mô hình ML cần có R² > {baseline_r2:.4f} và MAE < {baseline_mae:.4f} để có giá trị')

---
## 9. Mô hình 1 — Linear Regression <a id='9'></a>

In [ ]:
print('🔷 MÔ HÌNH 1: LINEAR REGRESSION')
print('=' * 50)
print('📖 Giải thích:')
print('  - Mô hình tuyến tính: ŷ = w₀ + w₁x₁ + w₂x₂ + ... + wₐxd')
print('  - Học tham số w bằng cách tối thiểu hóa MSE (OLS)')
print('  - Giả định: quan hệ tuyến tính giữa đặc trưng và giá')
print('  - Ưu điểm: Đơn giản, nhanh, dễ giải thích hệ số')
print('  - Nhược điểm: Không nắm bắt được quan hệ phi tuyến')

lr_p2 = LinearRegression()
lr_p2.fit(X_train_scaled_p2, y_train_p2)
y_pred_lr_p2 = lr_p2.predict(X_test_scaled_p2)

print(f'\nHệ số hồi quy (w):')
coef_df = pd.Series(lr_p2.coef_, index=feature_cols_p2).sort_values(key=abs, ascending=False)
print(coef_df.round(4))
print(f'Intercept (w₀): {lr_p2.intercept_:.4f}')

---
## 10. Mô hình 2 — Decision Tree Regressor <a id='10'></a>

In [ ]:
print('🔷 MÔ HÌNH 2: DECISION TREE REGRESSOR')
print('=' * 50)
print('📖 Giải thích:')
print('  - Phân chia đệ quy không gian đặc trưng thành các vùng')
print('  - Dự đoán = trung bình giá trong mỗi vùng lá')
print('  - Tiêu chí: Tối thiểu hóa MSE tại mỗi bước chia')
print('  - Ưu điểm: Phi tuyến, dễ giải thích, không cần chuẩn hóa')
print('  - Nhược điểm: Dễ overfitting')

dt_p2 = DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)
dt_p2.fit(X_train_p2, y_train_p2)
y_pred_dt_p2 = dt_p2.predict(X_test_p2)
print(f'\nSố lá cây: {dt_p2.get_n_leaves()}')

---
## 11. Mô hình 3 — Random Forest Regressor <a id='11'></a>

In [ ]:
print('🔷 MÔ HÌNH 3: RANDOM FOREST REGRESSOR')
print('=' * 50)
print('📖 Giải thích:')
print('  - Kết hợp nhiều Decision Tree (bagging + random features)')
print('  - Dự đoán cuối = trung bình các cây')
print('  - Ưu điểm: Giảm variance, ít overfitting hơn DT, cho feature importance')
print('  - Nhược điểm: Khó giải thích, chậm hơn DT đơn lẻ')

rf_p2 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
rf_p2.fit(X_train_p2, y_train_p2)
y_pred_rf_p2 = rf_p2.predict(X_test_p2)

# Feature importance
importances_p2 = pd.Series(rf_p2.feature_importances_, index=feature_cols_p2).sort_values(ascending=False)
print('\n📈 Feature Importance:')
print(importances_p2.round(4))

---
## 12. Mô hình 4 — Gradient Boosting Regressor <a id='12'></a>

In [ ]:
print('🔷 MÔ HÌNH 4: GRADIENT BOOSTING REGRESSOR')
print('=' * 50)
print('📖 Giải thích:')
print('  - Xây dựng tuần tự các cây yếu, mỗi cây học để sửa lỗi của cây trước')
print('  - Tối thiểu hóa loss function bằng gradient descent trong không gian hàm')
print('  - Ưu điểm: Thường đạt kết quả tốt nhất trong ML truyền thống')
print('  - Nhược điểm: Chậm hơn RF, nhiều siêu tham số cần điều chỉnh')

gbr_p2 = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                    max_depth=4, random_state=RANDOM_STATE)
gbr_p2.fit(X_train_p2, y_train_p2)
y_pred_gbr_p2 = gbr_p2.predict(X_test_p2)
print(f'\nSố estimators: {gbr_p2.n_estimators_}')

---
## 13. Mô hình 5 — Support Vector Regression <a id='13'></a>

In [ ]:
print('🔷 MÔ HÌNH 5: SUPPORT VECTOR REGRESSION (SVR)')
print('=' * 50)
print('📖 Giải thích:')
print('  - Tìm hàm f(x) sao cho mọi điểm trong "ε-tube" được coi là dự đoán đúng')
print('  - Sử dụng kernel RBF để xử lý quan hệ phi tuyến')
print('  - BẮT BUỘC phải chuẩn hóa feature trước khi dùng SVR')
print('  - Ưu điểm: Mạnh với dữ liệu nhỏ-trung bình, phi tuyến tốt')
print('  - Nhược điểm: Rất chậm với dataset lớn, khó tune')

svr_p2 = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr_p2.fit(X_train_scaled_p2, y_train_p2)
y_pred_svr_p2 = svr_p2.predict(X_test_scaled_p2)
print(f'\nSố support vectors: {len(svr_p2.support_)}')

---
## 14. Đánh giá Tổng hợp (5 độ đo) <a id='14'></a>

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error"""
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_regressor(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'Mô hình': name,
        'MAE':   round(mean_absolute_error(y_true, y_pred), 4),
        'MSE':   round(mse, 4),
        'RMSE':  round(np.sqrt(mse), 4),
        'R²':    round(r2_score(y_true, y_pred), 4),
        'MAPE%': round(mape(y_true, y_pred), 2),
    }

reg_results = [
    evaluate_regressor('Baseline (Mean)',                y_test_p2, y_pred_baseline_p2),
    evaluate_regressor('Linear Regression',              y_test_p2, y_pred_lr_p2),
    evaluate_regressor('Decision Tree Reg. (depth=6)',   y_test_p2, y_pred_dt_p2),
    evaluate_regressor('Random Forest Reg. (n=100)',     y_test_p2, y_pred_rf_p2),
    evaluate_regressor('Gradient Boosting Reg.',         y_test_p2, y_pred_gbr_p2),
    evaluate_regressor('SVR (RBF kernel)',               y_test_p2, y_pred_svr_p2),
]

reg_results_df = pd.DataFrame(reg_results).set_index('Mô hình')

print('📊 BẢNG ĐÁNH GIÁ — 5 ĐỘ ĐO — 5 MÔ HÌNH + BASELINE')
print('=' * 85)
print(reg_results_df.to_string())
print('=' * 85)
print()
print('📖 Giải thích các độ đo:')
print('  - MAE  (Mean Absolute Error)       : Sai số tuyệt đối trung bình (tỷ đồng). Dễ giải thích.')
print('  - MSE  (Mean Squared Error)        : Bình phương sai số. Phạt nặng outlier.')
print('  - RMSE (Root MSE)                  : Căn MSE. Cùng đơn vị với target (tỷ đồng).')
print('  - R²   (Coefficient of Determination): Tỉ lệ phương sai được giải thích. R²=1 là hoàn hảo.')
print('  - MAPE (Mean Absolute % Error)     : Sai số phần trăm. Dễ so sánh qua các dataset.')

In [ ]:
# Visualize kết quả
models_no_base = reg_results_df.drop('Baseline (Mean)', errors='ignore')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
metrics_to_plot = [
    ('MAE', 'MAE (tỷ đồng) — nhỏ hơn tốt hơn', True),
    ('RMSE', 'RMSE (tỷ đồng) — nhỏ hơn tốt hơn', True),
    ('R²', 'R² — lớn hơn tốt hơn (max=1)', False),
    ('MAPE%', 'MAPE% — nhỏ hơn tốt hơn', True),
]

for ax, (metric, title, lower_better) in zip(axes.flatten(), metrics_to_plot):
    vals = models_no_base[metric].values
    bars = ax.bar(range(len(models_no_base)), vals, color=colors_bar,
                  alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(models_no_base)))
    ax.set_xticklabels(['LR', 'DT', 'RF', 'GBR', 'SVR'], fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.4)
    
    # Highlight best
    best_idx = vals.argmin() if lower_better else vals.argmax()
    axes.flatten()[list(range(4))[list(metrics_to_plot).index((metric, title, lower_better))]] \
        .patches[best_idx].set_edgecolor('gold')
    axes.flatten()[list(range(4))[list(metrics_to_plot).index((metric, title, lower_better))]] \
        .patches[best_idx].set_linewidth(3)

plt.suptitle('So sánh 5 mô hình Regression theo 4 độ đo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_house_price/model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 15. So sánh Mô hình & Chọn mô hình tốt nhất <a id='15'></a>

In [ ]:
# Scatter plot: Giá thực tế vs Giá dự đoán
preds_dict = {
    'Linear\nRegression': y_pred_lr_p2,
    'Decision\nTree': y_pred_dt_p2,
    'Random\nForest': y_pred_rf_p2,
    'Gradient\nBoosting': y_pred_gbr_p2,
    'SVR': y_pred_svr_p2,
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors_scatter = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, (name, y_pred_temp) in enumerate(preds_dict.items()):
    r2_val = r2_score(y_test_p2, y_pred_temp)
    axes[i].scatter(y_test_p2, y_pred_temp, alpha=0.4, s=15, color=colors_scatter[i])
    min_val = min(y_test_p2.min(), y_pred_temp.min())
    max_val = max(y_test_p2.max(), y_pred_temp.max())
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Lý tưởng')
    axes[i].set_xlabel('Giá thực tế (tỷ)', fontsize=9)
    axes[i].set_ylabel('Giá dự đoán (tỷ)', fontsize=9)
    axes[i].set_title(f'{name}\nR²={r2_val:.3f}', fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=7)

plt.suptitle('Giá thực tế vs Giá dự đoán — 5 mô hình', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_house_price/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

# Chọn mô hình tốt nhất
best_model_p2_name = models_no_base['R²'].idxmax()
best_r2 = models_no_base.loc[best_model_p2_name, 'R²']
best_mae = models_no_base.loc[best_model_p2_name, 'MAE']

print(f'\n🏆 MÔ HÌNH TỐT NHẤT: {best_model_p2_name}')
print(f'   R²   = {best_r2:.4f}')
print(f'   MAE  = {best_mae:.4f} tỷ đồng')
print(f'   MAPE = {models_no_base.loc[best_model_p2_name, "MAPE%"]:.2f}%')
print(f'\n   Lý do chọn: R² cao nhất, MAE thấp nhất → dự đoán chính xác nhất')
print(f'   Cải thiện so với baseline R²: {best_r2 - r2_score(y_test_p2, y_pred_baseline_p2):.4f}')

In [ ]:
# Lưu mô hình tốt nhất
# Xác định mô hình tốt nhất
best_models_map = {
    'Linear Regression': (lr_p2, X_test_scaled_p2),
    'Decision Tree Reg. (depth=6)': (dt_p2, X_test_p2),
    'Random Forest Reg. (n=100)': (rf_p2, X_test_p2),
    'Gradient Boosting Reg.': (gbr_p2, X_test_p2),
    'SVR (RBF kernel)': (svr_p2, X_test_scaled_p2),
}

final_model_p2, _ = best_models_map.get(best_model_p2_name, (gbr_p2, X_test_p2))

joblib.dump(final_model_p2, 'part2_house_price/model_house_price.pkl')
joblib.dump(scaler_p2, 'part2_house_price/scaler_house.pkl')
joblib.dump(feature_cols_p2, 'part2_house_price/feature_cols_p2.pkl')
joblib.dump(df_ml, 'part2_house_price/df_encoded.pkl')

print(f'💾 Đã lưu mô hình: {best_model_p2_name}')
print('💾 Files: model_house_price.pkl, scaler_house.pkl, feature_cols_p2.pkl')

---
## 16. Ứng dụng — Demo Hệ thống <a id='16'></a>

In [ ]:
def predict_house_price(model, scaler, feature_values, use_scaling=True):
    """
    Chuyển đầu vào ứng dụng → biểu diễn đặc trưng → dự đoán giá nhà.
    Sử dụng CÙNG preprocessing pipeline với lúc training.
    """
    sample = np.array([feature_values])
    if use_scaling:
        sample = scaler.transform(sample)
    predicted_price = model.predict(sample)[0]
    return max(0, predicted_price)  # Giá không thể âm

# Cần biết thứ tự features
print(f'Thứ tự features: {feature_cols_p2}')

print('\n🏠 DEMO HỆ THỐNG DỰ ĐOÁN GIÁ NHÀ VIỆT NAM')
print('=' * 55)

# Lấy 3 mẫu thực từ tập test để demo
sample_indices = [0, len(X_test_p2)//2, -1]

for i, idx in enumerate(sample_indices):
    features = X_test_p2[idx]
    actual = y_test_p2[idx]
    
    # Dùng model phù hợp
    if 'Linear' in best_model_p2_name or 'SVR' in best_model_p2_name:
        features_input = X_test_scaled_p2[idx]
        predicted = predict_house_price(final_model_p2, scaler_p2, X_test_p2[idx], use_scaling=True)
    else:
        predicted = predict_house_price(final_model_p2, scaler_p2, features, use_scaling=False)
    
    error_pct = abs(predicted - actual) / actual * 100 if actual != 0 else 0
    
    print(f'\nBất động sản {i+1}:')
    for col, val in zip(feature_cols_p2, features):
        print(f'  {col}: {val:.1f}')
    print(f'  → Giá thực tế : {actual:.3f} tỷ đồng')
    print(f'  → Giá dự đoán : {predicted:.3f} tỷ đồng')
    print(f'  → Sai số      : {error_pct:.1f}%')

print('\n📍 Luồng xử lý:')
print('  Thông tin nhà → Feature vector ℝᵈ → (Chuẩn hóa) → ML Regressor → Giá dự đoán')

---
## 17. Reflection & Kết luận <a id='17'></a>

### Reflection

| Câu hỏi | Câu trả lời |
|---|---|
| Hệ thống nhận thông tin gì? | Các đặc trưng bất động sản (vị trí, diện tích, số phòng, loại nhà) |
| Biểu diễn nội tại? | Vector số thực x ∈ ℝᵈ sau mã hóa label và chuẩn hóa |
| Mô hình học gì? | Học quan hệ phi tuyến giữa đặc trưng và giá bất động sản |
| Dự đoán nào được tạo ra? | Giá ước tính liên tục (tỷ đồng) |
| Tại sao xử lý được đầu vào mới? | Mô hình tổng quát hóa các pattern đã học sang bất động sản mới |
| Hạn chế? | Không nắm bắt yếu tố vĩ mô (lãi suất, chính sách), dữ liệu có thể không đủ đa dạng |

### Kết luận

**Kết quả đạt được:**
- Xây dựng hoàn chỉnh pipeline hồi quy dự đoán giá nhà Việt Nam 2024
- Huấn luyện và so sánh 5 mô hình với 5 độ đo
- Mô hình tốt nhất đạt R² vượt trội so với baseline

**Phát hiện chính:**
- Diện tích là đặc trưng quan trọng nhất dự đoán giá
- Gradient Boosting / Random Forest thường tốt hơn LR do quan hệ phi tuyến mạnh
- Log-transform target có thể cải thiện hiệu suất (biến phân phối lệch → gần chuẩn hơn)

**Hướng cải thiện:**
- Thêm dữ liệu và đặc trưng (khoảng cách trung tâm, tiện ích xung quanh)
- Thử deep learning (MLP, TabNet) hoặc XGBoost với tuning tốt hơn
- Feature engineering: tương tác giữa diện tích × vị trí

In [ ]:
print('✅ HOÀN TẤT PHẦN 2 — HỆ DỰ ĐOÁN GIÁ NHÀ VIỆT NAM 2024')
print('=' * 60)
print(reg_results_df.to_string())